## Phase 5: Customer Lifetime Value (CLV) Modeling
**The Concept**: Standard churn prediction only tells us who might leave. CLV tells us how much money it will cost us if they do. We will use two specific probabilistic models from the lifetimes library:

1. BG/NBD Model: Predicts how many purchases a customer will make in the next 90 days.

2. Gamma-Gamma Model: Predicts the average monetary value of those future purchases.

Multiplying these two predictions together gives us the exact projected dollar value for every customer.

In [1]:
"""
Cell 1: Setup and Re-establishing the Time Wall
We load the raw data again and enforce our 90-day cutoff. We must train our financial models strictly on past behavior to predict the 90-day performance window.
"""

# Cell 1
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

# Define paths
CLEANED_DATA_PATH = '../data/processed/cleaned_online_retail.parquet'
ADVANCED_FEATURES_PATH = '../data/processed/advanced_features_and_churn.parquet'

print("Loading data for Phase 5...")
df_raw = pd.read_parquet(CLEANED_DATA_PATH)
df_base = pd.read_parquet(ADVANCED_FEATURES_PATH)

# Recreate the observation window (90 days) to prevent data leakage
max_date = df_raw['Date'].max()
cutoff_date = max_date - pd.Timedelta(days=90)
observation_data = df_raw[df_raw['Date'] < cutoff_date].copy()

print(f"Raw data shape: {df_raw.shape}")
print(f"Observation data shape: {observation_data.shape}")
print("Cell 1 complete. Ready for formatting.")

Loading data for Phase 5...
Raw data shape: (397885, 10)
Observation data shape: (236358, 10)
Cell 1 complete. Ready for formatting.


In [2]:
""""
Cell 2: Formatting Data for the lifetimes Library
The lifetimes library calculates Recency and Frequency differently than standard RFM. Here, Frequency means the number of repeat purchases, and Recency means the time between their first and last purchase. The library has a built-in function to calculate this perfectly from raw transactions.
"""

# Cell 2
from lifetimes.utils import summary_data_from_transaction_data

print("Formatting data for probabilistic modeling...")

# Transform raw transactions into the specific RFM format required by lifetimes
clv_data = summary_data_from_transaction_data(
    observation_data,
    customer_id_col='CustomerID',
    datetime_col='Date',
    monetary_value_col='TotalAmount',
    observation_period_end=cutoff_date
)

print(f"CLV Summary Data Shape: {clv_data.shape}")
display(clv_data.head())

Formatting data for probabilistic modeling...
CLV Summary Data Shape: (3370, 4)


,frequency,recency,T,monetary_value
CustomerID,,,,
12346,0.0,0.0,235.0,0.0000
12347,4.0,238.0,277.0,519.7675
12348,2.0,110.0,268.0,297.2200
12350,0.0,0.0,220.0,0.0000
12352,3.0,34.0,206.0,421.7700


In [3]:
"""
Cell 3: Fitting the BG/NBD Model
This algorithm looks at the gaps between past purchases to calculate the probability that a customer is still "alive" and predicts how many times they will buy in the next 90 days.

"""

# Cell 3
from lifetimes import BetaGeoFitter

print("Fitting the BG/NBD Model (Predicting transaction volume)...")

# Dynamic auto-convergence loop
penalizer_val = 0.001
bgf = None

while penalizer_val <= 1.0:
    try:
        print(f"Attempting to fit with penalizer_coef: {penalizer_val:.3f}...")
        bgf = BetaGeoFitter(penalizer_coef=penalizer_val)
        bgf.fit(clv_data['frequency'], clv_data['recency'], clv_data['T'])
        print(f"SUCCESS! BG/NBD Model converged at penalizer: {penalizer_val:.3f}")
        break # Exit loop once successful
    except:
        penalizer_val += 0.05 # Increase the penalizer and try again

if bgf is None:
    raise ValueError("Critical Error: Model failed to converge completely.")

# Predict the number of purchases in our 90-day window
t = 90
clv_data['predicted_purchases_90d'] = bgf.conditional_expected_number_of_purchases_up_to_time(
    t, clv_data['frequency'], clv_data['recency'], clv_data['T']
)

display(clv_data[['frequency', 'recency', 'T', 'predicted_purchases_90d']].head())

Fitting the BG/NBD Model (Predicting transaction volume)...
Attempting to fit with penalizer_coef: 0.001...
  message: Desired error not necessarily achieved due to precision loss.
  success: False
   status: 2
      fun: nan
        x: [-6.721e-01 -1.879e+00 -4.571e+03 -3.692e+03]
      nit: 31
      jac: [       nan        nan        nan        nan]
 hess_inv: [[ 2.995e+00  2.511e+00  8.765e+01  6.427e+01]
            [ 2.511e+00  3.667e+00 -2.187e+01 -2.605e+01]
            [ 8.765e+01 -2.187e+01  1.317e+07  1.064e+07]
            [ 6.427e+01 -2.605e+01  1.064e+07  8.602e+06]]
     nfev: 142
     njev: 142
Attempting to fit with penalizer_coef: 0.051...
SUCCESS! BG/NBD Model converged at penalizer: 0.051


,frequency,recency,T,predicted_purchases_90d
CustomerID,,,,
12346,0.0,0.0,235.0,0.184499
12347,4.0,238.0,277.0,1.249169
12348,2.0,110.0,268.0,0.724841
12350,0.0,0.0,220.0,0.194605
12352,3.0,34.0,206.0,1.244132


In [4]:
"""
Cell 4: Fitting the Gamma-Gamma Model
Now we predict the financial value. The Gamma-Gamma model requires that a customer has made at least one repeat purchase (Frequency > 0) to understand their spending habits.

"""
# Cell 4
from lifetimes import GammaGammaFitter
import numpy as np

print("Fitting the Gamma-Gamma Model (Predicting monetary value)...")

# The Gamma-Gamma model strictly requires customers with at least one repeat purchase
returning_customers = clv_data[clv_data['frequency'] > 0]

# Dynamic auto-convergence loop for Gamma-Gamma
gg_penalizer = 0.001
ggf = None

while gg_penalizer <= 1.0:
    try:
        print(f"Attempting to fit with penalizer_coef: {gg_penalizer:.3f}...")
        ggf = GammaGammaFitter(penalizer_coef=gg_penalizer)
        ggf.fit(returning_customers['frequency'], returning_customers['monetary_value'])
        print(f"SUCCESS! Gamma-Gamma Model converged at penalizer: {gg_penalizer:.3f}")
        break
    except:
        gg_penalizer += 0.05

if ggf is None:
    raise ValueError("Critical Error: Model failed to converge completely.")

# Predict average order value for all customers
clv_data['predicted_aov'] = ggf.conditional_expected_average_profit(
    clv_data['frequency'], clv_data['monetary_value']
)

# THE FIX: Overwrite any anomalies (like negatives or NaNs) for 0-frequency customers
# We give them the baseline average of returning customers
overall_average_value = returning_customers['monetary_value'].mean()
clv_data['predicted_aov'] = np.where(
    clv_data['frequency'] == 0, 
    overall_average_value, 
    clv_data['predicted_aov']
)

print("\nGamma-Gamma Model trained and anomalies fixed.")
display(clv_data[['frequency', 'monetary_value', 'predicted_aov']].head())

Fitting the Gamma-Gamma Model (Predicting monetary value)...
Attempting to fit with penalizer_coef: 0.001...
SUCCESS! Gamma-Gamma Model converged at penalizer: 0.001

Gamma-Gamma Model trained and anomalies fixed.


,frequency,monetary_value,predicted_aov
CustomerID,,,
12346,0.0,0.0000,418.376319
12347,4.0,519.7675,524.431725
12348,2.0,297.2200,305.000390
12350,0.0,0.0000,418.376319
12352,3.0,421.7700,427.533229


In [5]:
"""Cell 5: Calculating the 90-Day CLV
We simply multiply the expected number of purchases by the expected value of those purchases."""
# Cell 5
print("Calculating 90-Day Customer Lifetime Value (CLV)...")

# Calculate final predicted revenue (Volume * Value)
clv_data['predicted_90d_clv'] = clv_data['predicted_purchases_90d'] * clv_data['predicted_aov']

# Sort to view the "Whales" (most valuable customers)
top_customers = clv_data.sort_values(by='predicted_90d_clv', ascending=False)

print("\n--- Top 5 Most Valuable Customers (Projected Next 90 Days) ---")
display(top_customers[['frequency', 'predicted_purchases_90d', 'predicted_aov', 'predicted_90d_clv']].head())


Calculating 90-Day Customer Lifetime Value (CLV)...

--- Top 5 Most Valuable Customers (Projected Next 90 Days) ---


,frequency,predicted_purchases_90d,predicted_aov,predicted_90d_clv
CustomerID,,,,
14646,29.0,8.378965,6149.299390,51524.765195
18102,15.0,4.241419,7309.651639,31003.297137
12415,11.0,3.467765,8655.443942,30015.048472
14156,30.0,8.222353,3286.458946,27022.424630
17450,16.0,4.513442,5282.405667,23841.831443


In [6]:
"""
Cell 6: The Grand Merge and Save
We take this newly calculated predicted_90d_clv and merge it back into the master dataset containing our Churn labels and advanced behavioral features.
"""
# Cell 6
import os

print("Merging CLV predictions into the master dataset...")

# Extract only the final CLV calculation
clv_subset = clv_data[['predicted_90d_clv']].reset_index()

# Merge into our advanced features dataset (df_base from Phase 3/4)
df_master = pd.merge(df_base, clv_subset, on='CustomerID', how='inner')

# Save the ultimate master dataset
MASTER_DATA_PATH = '../data/processed/master_customer_dataset.parquet'

# Ensure directory exists just in case
os.makedirs('../data/processed', exist_ok=True)

df_master.to_parquet(MASTER_DATA_PATH, index=False)

print(f"Master Dataset successfully saved to: {MASTER_DATA_PATH}")
print(f"Final Master Shape: {df_master.shape}")


Merging CLV predictions into the master dataset...
Master Dataset successfully saved to: ../data/processed/master_customer_dataset.parquet
Final Master Shape: (3370, 10)
